# Demo: Run a Classifier Directly on Images

### Objective
Infer and evaluate lesion classes on RCC-AID scans

### Prerequisites:
- RenalVision

## Data Download
You can download the pre-annotated RCC-AID dataset from zenodo: https://zenodo.org/records/20719257

Each image includes exactly one solid tumour. Some images additionaly have one or multiple cysts.

It has the following annotations:

- 1 : Kidney
- 2 : Tumour
- 3 : Cyst

---

## Binary Classification (Solid / Cystic)

In [ ]:
from pathlib import Path

# Directory where you downloaded RCC-AID to:
basepath = Path("/your/custom/path/to/rcc-aid")

In [ ]:
from tqdm.auto import tqdm

from renal_vision.modeling.inference import LesionPredictor

# 1. Initialise your prediction model
predictor = LesionPredictor("RADIOMICS_BINARY")

# If you have trained your own model, for example on KiTS, you can specify it directly
# predictor = LesionPredictor("model/model.pkl")


# 2. Define Labelmap
labelmap = {
    1:-1, # exclude kidney
    2:0, # First class: tumours
    3:1, # Second class: cysts
}


# 3. Define your dataset
img_path = basepath / "images"
seg_path = basepath / "labels"
images =  [p.name for p in img_path.glob("*.nii.gz")]
images = [img for img in images if (seg_path / img.replace(".nii","_seg.nii")).exists()]
print(f"Found {len(images)} with corresponding annotations")


# 4. Iterate over Images
y_true = []
y_prob = []
for img in tqdm(images):

    # filter for components of size >= 400m^3
    seg_filtered, metadata_list = predictor.filter_components(seg_path / img.replace(".nii","_seg.nii"))
    
    for lesion_id, metadata in enumerate(metadata_list, start=1):
        
        # skip ignored classes (kidney)
        true_class_id = labelmap[metadata["class_id"]]
        if true_class_id == -1:
            continue

        # Run inference
        prediction = predictor.infer_lesion(img_path / img,seg_filtered==lesion_id)
        tqdm.write(f"Groundtruth: {true_class_id} \t Prediction: {prediction}")

        y_true.append(true_class_id)
        y_prob.append(prediction["probability"])

### Evaluate

In [ ]:
from renal_vision.shared.metrics import ModelEvaluator

class_names = ["Tumour", "Cyst"]
evaluator = ModelEvaluator(y_true, y_prob, class_names)

results = evaluator.get_scalars()
auc = evaluator.get_auc(with_ci=False)
ap = evaluator.get_ap(with_ci=False)
print(results)
print("AUC:", auc)
print("AP:", ap)

evaluator.plot_cm(figsize=(6,5))
evaluator.plot_roc(figsize=(6,5), show_ci = True, show_grid = False)
evaluator.plot_pr(figsize=(6,5), show_grid = False)
evaluator.plot_pr_grid(show_ci = True, show_grid = False)

## Multi-Class Classification
Each image includes exactly one solid tumour. Some images additionaly have one or multiple cysts.
The specific histologic subtypes of the solid tumours can be deducted from the prefix in the image name:

- Cysts
- clear cell Renal Cell Carcinoma (KIRC)
- pailarry Renal Cell Carcinoma (KIRP)
- chromophobe Renal Cell Carcinoma (KICH)

In [ ]:
from tqdm.auto import tqdm

from renal_vision.bundles import ImplementedModels
from renal_vision.modeling.inference import LesionPredictor

# 1. Initialise your prediction model
predictor = LesionPredictor(model_identifier=ImplementedModels.RADIOMICS)

# 2. Define Labelmap
labelmap = {
    1:-1, # exclude kidney
    2:0, # First class: tumours
    3:1, # Second class: cysts
}

# as defined in the radiomics base classifier
multiclass_map = {
    "KIRC": 0,
    "KIRP": 1,
    "KICH": 2,
    "Oncocytoma:": 3, # (not in RCC-AID)
    "Cyst": 4,
    "Other": 5 # (not in RCC-AID)
}

# 3. Define your dataset
img_path = basepath / "images"
seg_path = basepath / "labels"
images =  [p.name for p in img_path.glob("*.nii.gz")]
images = [img for img in images if (seg_path / img.replace(".nii","_seg.nii")).exists()]
print(f"Found {len(images)} with corresponding annotations")


# 4. Iterate over Images
y_true = []
y_prob = []
for img in tqdm(images):

    # filter for components of size >= 400m^3
    seg_filtered, metadata_list = predictor.filter_components(seg_path / img.replace(".nii","_seg.nii"))
    
    for lesion_id, metadata in enumerate(metadata_list, start=1):
        
        # skip ignored classes (kidney)
        true_class_id = labelmap[metadata["class_id"]]
        if true_class_id == -1:
            continue

        # New: asign cysts and solid classes to their specific histological subtype
        if true_class_id == 0:
            true_class_id = multiclass_map[img.split("_")[0]] # asign tumor type based on image name prefix
        elif true_class_id == 1:
            true_class_id = multiclass_map["Cyst"]
        else:
            raise ValueError("This should not happen")

        # Run inference
        prediction = predictor.infer_lesion(img_path / img,seg_filtered==lesion_id)
        tqdm.write(f"Groundtruth: {true_class_id} \t Prediction: {prediction}")

        y_true.append(true_class_id)
        y_prob.append(prediction["probability"])

In [ ]:
from renal_vision.shared.metrics import ModelEvaluator

class_names = list(predictor.bundle.class_names.values())
evaluator = ModelEvaluator(y_true, y_prob, class_names)

results = evaluator.get_scalars()
auc = evaluator.get_auc(with_ci=False)
ap = evaluator.get_ap(with_ci=False)
print(results)
print("AUC:", auc)
print("AP:", ap)

evaluator.plot_cm(figsize=(6,5))
evaluator.plot_roc(figsize=(6,5), show_ci = True, show_grid = False)
evaluator.plot_pr(figsize=(6,5), show_grid = False)
evaluator.plot_pr_grid(show_ci = True, show_grid = False)